# 🛰️ Competitor Intelligence Radar: Dynamic Competitive Analysis

### Overview
In Week 1, we explored single-page web summarization using standard HTTP scrapers and Frontier LLMs. However, modern enterprise and developer platforms (such as OpenAI, Anthropic, or Stripe) rely heavily on client-side JavaScript rendering[cite: 8], which causes standard `requests`-based scrapers to return blank shells or 403 blocks[cite: 8].

**Competitor Intelligence Radar** extends this concept into a practical multi-site competitive analysis engine:
1. **Headless Dynamic Extraction:** Uses Playwright to render JavaScript single-page applications and clean out navigation, tracking, and styling noise.
2. **Dual-Entity Competitive Matrix:** Extracts live messaging from two competing companies simultaneously.
3. **Structured Executive Intelligence:** Instructs an LLM to generate an executive battlecard, a strategic differentiator teardown, and a snarky final verdict.


## Step 1: Environment Setup & Resilient Client Initialization

We load credentials from `.env`[cite: 6, 8]. The setup supports both standard OpenAI API keys and OpenRouter API keys transparently, falling back to free tier models if needed[cite: 8].


In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display
from radar_scraper import fetch_dynamic_content

# Load local environment variables
load_dotenv(override=True)

# Detect API configuration (OpenRouter or direct OpenAI)
api_key = os.getenv("OPENROUTER_API_KEY") or os.getenv("OPENAI_API_KEY")
base_url = "https://openrouter.ai/api/v1" if os.getenv("OPENROUTER_API_KEY") else None

if not api_key:
    raise ValueError("No API key found! Please set OPENROUTER_API_KEY or OPENAI_API_KEY in your .env file.")

client = OpenAI(
    api_key=api_key,
    base_url=base_url
)

# Route to active model target
MODEL_NAME = "openrouter/free" if base_url else "gpt-4o-mini"
print(f"✅ Initialized client using model target: {MODEL_NAME}")

## Step 2: Competitive Intelligence Prompt Design

To produce actionable insights rather than generic text summaries, the prompt enforces a structured Markdown schema:
- **Executive Battlecard:** A concise side-by-side comparison table covering target audience, core pitch, and key claims.
- **Strategic Analysis:** Identifies what each competitor highlights and pinpoints omissions (the "blindspot").
- **Final Verdict:** Retains the punchy, snarky tone from the Week 1 lab[cite: 8].

In [ ]:
SYSTEM_RADAR_PROMPT = """
You are an elite Competitive Intelligence Strategist. 
You will be provided with scraped content from TWO competing companies.
Perform an objective, razor-sharp teardown comparing both companies in GitHub-flavored Markdown.

Your teardown MUST follow this exact structure:
##  Executive Battlecard
| Metric | Company A | Company B |
| :--- | :--- | :--- |
| **Target Audience** | ... | ... |
| **Primary Value Pitch** | ... | ... |
| **Key Tech / Product Claim** | ... | ... |

##  Strategic Analysis
- **Company A Core Focus:** ...
- **Company B Core Focus:** ...
- **The Blindspot:** (What both are conspicuously avoiding saying or downplaying)

##  Final Verdict
A humorous, snarky summary declaring which product makes a clearer, more credible business case to developers/executives.
"""

async def generate_competitor_radar(url_a: str, url_b: str):
    print(f" [1/3] Scraping Company A: {url_a}...")
    content_a = await fetch_dynamic_content(url_a, headless=False)
    
    print(f" [2/3] Scraping Company B: {url_b}...")
    content_b = await fetch_dynamic_content(url_b, headless=False)

    user_payload = f"""
    Please generate the competitive battlecard. Do NOT output JSON or function calls. Output only GitHub-flavored Markdown text.

    === COMPANY A ({url_a}) ===
    {content_a[:3500]}

    === COMPANY B ({url_b}) ===
    {content_b[:3500]}
    """

    messages = [
        {"role": "system", "content": SYSTEM_RADAR_PROMPT},
        {"role": "user", "content": user_payload}
    ]

    print(f" [3/3] Generating report with {MODEL_NAME}...\n")
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages
    )

    display(Markdown(response.choices[0].message.content))

## Step 3: Live Radar Execution: Anthropic vs. OpenAI

We test our Playwright-driven radar on two JavaScript-heavy frontier AI platforms: `https://anthropic.com` and `https://openai.com`[cite: 8].

In [ ]:
await generate_competitor_radar("https://anthropic.com", "https://openai.com")

## 💼 Business Applications & Engineering Takeaways

1. **Automation of Market Research:** Replaces hours of manual review during product teardowns or investment diligence by pulling directly from live public landing pages.
2. **Resilience to Single-Page Applications (SPAs):** Using Playwright guarantees that dynamic client-side DOM frameworks (React, Next.js, Vue) render properly before passing data to the LLM.
3. **Structured Outputs:** By enforcing table schemas and specific section headers in the system prompt, the output can be piped directly into internal developer portals, Slack bots, or CRM systems.